# Lab 7 — Medallion Architecture: Bronze to Silver Cleansing & Deduplication

**Goal:** Read Bronze as a stream, filter invalid records, apply a watermark, deduplicate, and write Silver.

**Prerequisite:** Run Lab 6 first so `workspace.lab_db.bronze_orders_lab6` exists. This lab uses a dedicated Silver table/checkpoint to avoid colliding with other streams.

In [0]:

from pyspark.sql.functions import col, to_timestamp, expr

BRONZE_TABLE = "workspace.lab_db.bronze_orders_lab6"
SILVER_TABLE = "workspace.lab_db.silver_orders_lab7"

BASE_PATH = "/Volumes/workspace/lab_db/lab06_files"
CHECKPOINT = f"{BASE_PATH}/checkpoints/silver_orders"

# Read Bronze as a streaming DataFrame
bronze = spark.readStream.table(BRONZE_TABLE)

# Explicitly convert the incoming data types
silver = (
    bronze
    .withColumn("amount", expr("try_cast(amount AS DOUBLE)"))
    .withColumn("customer_id", expr("try_cast(customer_id AS INT)"))
    .withColumn("order_timestamp", to_timestamp(col("order_timestamp")))
    .filter(
        col("customer_id").isNotNull()
        & col("amount").isNotNull()
        & (col("amount") > 0)
        & col("order_id").isNotNull()
        & col("order_timestamp").isNotNull()
    )
    .withWatermark("order_timestamp", "10 minutes")
    .dropDuplicates(["order_id"])
)

# Write to Silver
query = (
    silver.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(SILVER_TABLE)
)

query.awaitTermination()
print("Silver stream completed successfully!")


In [0]:
display(spark.table("workspace.lab_db.silver_orders_lab7").orderBy("order_id"))

In [0]:

from pyspark.sql.functions import col, expr

silver_df = spark.table("workspace.lab_db.silver_orders_lab7")

# Convert values to suitable types for validation
checked_df = (
    silver_df
    .withColumn("amount_num", expr("try_cast(amount AS DOUBLE)"))
    .withColumn("customer_id_num", expr("try_cast(customer_id AS INT)"))
)

invalid_customer_count = (
    checked_df.filter(col("customer_id_num").isNull()).count()
)

invalid_amount_count = (
    checked_df.filter(
        col("amount_num").isNull() | (col("amount_num") <= 0)
    ).count()
)

duplicate_order_count = (
    checked_df.groupBy("order_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

results = {
    "silver_rows": checked_df.count(),
    "invalid_customer_rows": invalid_customer_count,
    "invalid_amount_rows": invalid_amount_count,
    "duplicate_order_ids": duplicate_order_count
}

print(results)

assert invalid_customer_count == 0, "Invalid customer IDs found"
assert invalid_amount_count == 0, "Invalid amounts found"
assert duplicate_order_count == 0, "Duplicate order IDs found"

print("All data quality checks passed!")


**Note:** Watermark-based deduplication is stateful and only guarantees deduplication within the watermark/state-retention semantics. For a one-time batch dataset, exact global deduplication can also be demonstrated with `dropDuplicates` on a batch DataFrame.